In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

hourly_df = pd.read_csv('../data/processed/hourly_energy.csv')
hourly_df['timestamp'] = pd.to_datetime(hourly_df['timestamp'])
hourly_df.head()

,timestamp,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,date,year,month,hour,day_of_week,is_weekend
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,19.0,607.0,2006-12-16,2006,12,17,Saturday,True
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0,2006-12-16,2006,12,18,Saturday,True
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0,2006-12-16,2006,12,19,Saturday,True
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0,2006-12-16,2006,12,20,Saturday,True
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0,2006-12-16,2006,12,21,Saturday,True


In [2]:
mean_consumption = hourly_df['Global_active_power'].mean()
std_consumption = hourly_df['Global_active_power'].std()

hourly_df['z_score'] = (hourly_df['Global_active_power'] - mean_consumption) / std_consumption

print(f"Mean consumption: {mean_consumption:.2f} kWh")
print(f"Standard deviation: {std_consumption:.2f} kWh")

Mean consumption: 1.09 kWh
Standard deviation: 0.90 kWh


In [3]:
hourly_df['is_anomaly'] = hourly_df['z_score'].abs() > 2.5

anomaly_count = hourly_df['is_anomaly'].sum()
print(f"Total anomalies detected: {anomaly_count}")
print(f"Percentage of data flagged as anomaly: {(anomaly_count/len(hourly_df))*100:.2f}%")

Total anomalies detected: 862
Percentage of data flagged as anomaly: 2.49%


In [4]:
anomalies = hourly_df[hourly_df['is_anomaly']].sort_values('Global_active_power', ascending=False)
anomalies[['timestamp', 'Global_active_power', 'z_score', 'hour', 'day_of_week']].head(10)

,timestamp,Global_active_power,z_score,hour,day_of_week
16993,2008-11-23 18:00:00,6.560533,6.092569,18,Sunday
18291,2009-01-16 20:00:00,6.519633,6.047004,20,Friday
9914,2008-02-02 19:00:00,6.496033,6.020712,19,Saturday
8930,2007-12-23 19:00:00,6.488000,6.011762,19,Sunday
1636,2007-02-22 21:00:00,6.363867,5.873470,21,Thursday
9048,2007-12-28 17:00:00,6.333667,5.839826,17,Friday
16995,2008-11-23 20:00:00,6.310567,5.814091,20,Sunday
867,2007-01-21 20:00:00,6.076567,5.553401,20,Sunday
9746,2008-01-26 19:00:00,6.013800,5.483476,19,Saturday
17163,2008-11-30 20:00:00,5.930500,5.390675,20,Sunday


In [5]:
def explain_anomaly(row):
    expected = mean_consumption
    actual = row['Global_active_power']
    diff_pct = ((actual - expected) / expected) * 100
    
    return f"{row['timestamp']}: Consumption was {actual:.2f} kWh, {diff_pct:.1f}% above the average of {expected:.2f} kWh. This occurred at {row['hour']}:00 on a {row['day_of_week']}."

top_anomalies = anomalies.head(5)
for idx, row in top_anomalies.iterrows():
    print(explain_anomaly(row))
    print()

2008-11-23 18:00:00: Consumption was 6.56 kWh, 500.9% above the average of 1.09 kWh. This occurred at 18:00 on a Sunday.

2009-01-16 20:00:00: Consumption was 6.52 kWh, 497.2% above the average of 1.09 kWh. This occurred at 20:00 on a Friday.

2008-02-02 19:00:00: Consumption was 6.50 kWh, 495.0% above the average of 1.09 kWh. This occurred at 19:00 on a Saturday.

2007-12-23 19:00:00: Consumption was 6.49 kWh, 494.3% above the average of 1.09 kWh. This occurred at 19:00 on a Sunday.

2007-02-22 21:00:00: Consumption was 6.36 kWh, 482.9% above the average of 1.09 kWh. This occurred at 21:00 on a Thursday.



In [6]:
hourly_df.to_csv('../data/processed/hourly_energy_with_anomalies.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [7]:
hourly_stats = hourly_df.groupby('hour')['Global_active_power'].agg(['mean', 'std']).reset_index()
hourly_stats.columns = ['hour', 'hour_mean', 'hour_std']

hourly_df = hourly_df.merge(hourly_stats, on='hour')

hourly_df['z_score_by_hour'] = (hourly_df['Global_active_power'] - hourly_df['hour_mean']) / hourly_df['hour_std']

hourly_df['is_anomaly_v2'] = hourly_df['z_score_by_hour'].abs() > 2.5

print(f"Total anomalies (hour-adjusted): {hourly_df['is_anomaly_v2'].sum()}")

Total anomalies (hour-adjusted): 855


In [8]:
anomalies_v2 = hourly_df[hourly_df['is_anomaly_v2']].sort_values('z_score_by_hour', ascending=False)
anomalies_v2[['timestamp', 'Global_active_power', 'hour', 'hour_mean', 'z_score_by_hour', 'day_of_week']].head(10)

,timestamp,Global_active_power,hour,hour_mean,z_score_by_hour,day_of_week
16136,2008-10-19 01:00:00,5.759067,1,0.539325,10.416596,Sunday
227,2006-12-26 04:00:00,2.992500,4,0.443844,7.400440,Tuesday
1161,2007-02-03 02:00:00,3.498267,2,0.480618,7.354663,Saturday
16135,2008-10-19 00:00:00,5.155500,0,0.659562,7.277057,Sunday
1184,2007-02-04 01:00:00,4.159333,1,0.539325,7.224144,Sunday
1183,2007-02-04 00:00:00,5.073333,0,0.659562,7.144063,Sunday
33948,2010-10-31 05:00:00,2.910300,5,0.453674,6.934343,Sunday
1570,2007-02-20 03:00:00,2.847333,3,0.444850,6.727231,Tuesday
1520,2007-02-18 01:00:00,3.888367,1,0.539325,6.683398,Sunday
9415,2008-01-13 00:00:00,4.670200,0,0.659562,6.491558,Sunday


In [9]:
hourly_df.to_csv('../data/processed/hourly_energy_with_anomalies.csv', index=False)
print("Saved successfully!")

Saved successfully!
